In [1]:
import os
import sys

# Uninstall semua package yang bermasalah
!pip uninstall -y transformers tokenizers huggingface-hub

# Install versi stabil yang kompatibel
!pip install transformers==4.51.0 tokenizers==0.21.4 huggingface-hub==0.36.2

# Install library pendukung
!pip install accelerate bitsandbytes
!pip install pyngrok fastapi uvicorn nest_asyncio

# Bersihkan cache
!rm -rf ~/.cache/huggingface

print("[SUCCESS] Transformers 4.51.0 terinstall. Silakan restart kernel sebelum melanjutkan ke CELL 2.")

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: huggingface_hub 1.11.0
Uninstalling huggingface_hub-1.11.0:
  Successfully uninstalled huggingface_hub-1.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 77.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.9 MB/s eta 0:00:00:00:0100:01
[SUCCESS] Transformers 4.51.0 terinstall. Silakan restart kernel sebelum melanjutkan ke CELL 2.


In [9]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_LLM")

#### Model SmolStruct-1.7B

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

import json
import torch
import re
import threading
import time
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

print("="*60)
print("[STARTING] Encoder Server (Qwen2.5-14B-Instruct 4-bit) - MULTILINGUAL ROOM EXTRACTION")
print("="*60)

# --- Kill port 8000 ---
os.system("fuser -k 8000/tcp 2>/dev/null || true")
time.sleep(1)
print("[SERVER] Port 8000 cleared.")

# --- ngrok token ---
user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_LLM")
print("[SERVER] ngrok token loaded")

# --- Load Model (4-bit) ---
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
print(f"[LOAD] Loading model {MODEL_ID}...")

try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    model.eval()
    print("[SUCCESS] Model loaded successfully!")
    print(f"   - Device: {model.device}")
    print(f"   - Parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
except Exception as e:
    print(f"[FAILED] Error loading model: {e}")
    raise

# ============================================================
# JSON SCHEMA UNTUK ROOM DETAIL
# ============================================================
ROOM_SCHEMA = {
    "type": "object",
    "properties": {
        "rooms": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "type": {
                        "type": "string",
                        "enum": ["bedroom", "bathroom", "living_room", "kitchen", "balcony", "common_room", "storage", "dining_room"]
                    },
                    "size": {
                        "type": "object",
                        "properties": {
                            "width": {"type": "number", "minimum": 1},
                            "height": {"type": "number", "minimum": 1}
                        },
                        "required": ["width", "height"]
                    },
                    "location": {"type": "string"},
                    "links": {"type": "array", "items": {"type": "string"}}
                },
                "required": ["name", "type", "size"]
            }
        },
        "summary": {
            "type": "object",
            "properties": {
                "bedroom": {"type": "integer", "minimum": 0},
                "bathroom": {"type": "integer", "minimum": 0},
                "living_room": {"type": "integer", "minimum": 0},
                "kitchen": {"type": "integer", "minimum": 0},
                "balcony": {"type": "integer", "minimum": 0},
                "common_room": {"type": "integer", "minimum": 0}
            }
        }
    },
    "required": ["rooms"]
}

def build_chat_messages(user_text: str):
    system_prompt = """You are an expert architectural assistant that extracts room details from floor plan descriptions.

Your task is to analyze the user's description and output a structured JSON with two main sections:

1. **"rooms"**: A list of each room mentioned with these exact fields:
   - "name": The room's name exactly as described (e.g., "master bedroom", "bathroom 1", "living room")
   - "type": One of: bedroom, bathroom, living_room, kitchen, balcony, common_room, storage, dining_room
   - "size": An object with "width" and "height" in feet (if not explicitly mentioned, use reasonable defaults based on room type)
   - "location": The room's position (e.g., "north", "southwest", "center", "east side", "northwest corner")
   - "links": An array of room names that this room connects to or is adjacent to

2. **"summary"**: A summary count of each room type (bedroom, bathroom, living_room, kitchen, balcony, common_room)

IMPORTANT GUIDELINES:
- This is a MULTILINGUAL system. The user may describe rooms in English, Indonesian, or mixed languages. Understand the meaning regardless of language.
- Count rooms accurately from the description (e.g., "2 kamar tidur" = 2 bedrooms, "satu kamar mandi" = 1 bathroom).
- For location, interpret directional words in any language:
  - utara/north, selatan/south, timur/east, barat/west, tengah/center, pojok/corner, samping/side
- If location is not specified, use "center".
- For links/connections, look for words like: connected to, adjacent to, next to, near, beside, sebelah, bersebelahan, dekat, terhubung dengan
- If links are not mentioned, leave empty array [].
- For size, look for patterns like "10x15 ft", "10 feet by 15 feet", "10 x 15", "lebar 10 panjang 15" (Indonesian).
- If size is not specified, use these defaults:
  - bedroom: 12x15
  - living_room: 20x15
  - kitchen: 10x10
  - bathroom: 7x5
  - balcony: 10x5
  - common_room: 15x12
  - storage: 8x8
  - dining_room: 12x10

MULTILINGUAL ROOM TYPE MAPPING (CRITICAL - FOLLOW THIS EXACTLY):
- "ruang utama", "master room", "kamar utama", "master bedroom", "kamar tidur utama" → type: "bedroom"
- "kamar tidur", "bedroom", "kamar" (if alone as a room) → type: "bedroom"
- "ruang tamu", "living room", "ruang keluarga", "family room", "lounge" → type: "living_room"
- "kamar mandi", "bathroom", "toilet", "bath" → type: "bathroom"
- "dapur", "kitchen", "pantry" → type: "kitchen"
- "balkon", "balcony", "terrace", "veranda" → type: "balcony"
- "ruang bersama", "common room", "ruang serbaguna", "multi-purpose room" → type: "common_room"
- "ruang penyimpanan", "storage", "gudang" → type: "storage"
- "ruang makan", "dining room", "dining" → type: "dining_room"

OUTPUT MUST BE VALID JSON ONLY. No additional text, no explanations, no markdown.

EXAMPLE OUTPUT:
{
  "rooms": [
    {"name": "master bedroom", "type": "bedroom", "size": {"width": 12, "height": 15}, "location": "southwest", "links": ["living room"]},
    {"name": "bathroom 1", "type": "bathroom", "size": {"width": 7, "height": 5}, "location": "north", "links": ["living room"]},
    {"name": "living room", "type": "living_room", "size": {"width": 20, "height": 15}, "location": "west", "links": ["kitchen", "bathroom 1"]},
    {"name": "kitchen", "type": "kitchen", "size": {"width": 10, "height": 10}, "location": "north", "links": ["living room"]}
  ],
  "summary": {
    "bedroom": 1,
    "bathroom": 1,
    "living_room": 1,
    "kitchen": 1,
    "balcony": 0,
    "common_room": 0
  }
}

Now extract the room details from this description:"""
    user_prompt = f'Description: "{user_text}"\nOutput JSON:'
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

def try_llm_extract_detailed(user_text: str):
    """
    Ekstraksi detail ruangan langsung dari LLM (tanpa regex fallback).
    """
    messages = build_chat_messages(user_text)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_tokens = inputs.input_ids.shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    generated_ids = outputs[0][input_tokens:]
    output_tokens = generated_ids.shape[0]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    print(f"🔍 [DEBUG] Raw LLM output:\n{raw_output}\n{'-'*50}")
    
    # Bersihkan output: ambil JSON murni
    json_str = raw_output.strip()
    
    # Hilangkan prefix/suffix yang tidak diinginkan
    if json_str.startswith("assistant"):
        json_str = json_str[len("assistant"):].strip()
    
    # Cari objek JSON
    match = re.search(r'\{.*\}', json_str, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in LLM output")
    
    json_str = match.group(0)
    
    # Parse JSON
    try:
        data = json.loads(json_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {e}")
    
    # Validasi minimal: harus ada 'rooms'
    if "rooms" not in data or not data["rooms"]:
        raise ValueError("JSON missing 'rooms' array")
    
    # Validasi setiap room punya field wajib
    for room in data["rooms"]:
        if "name" not in room:
            raise ValueError("Room missing 'name'")
        if "type" not in room:
            raise ValueError(f"Room '{room.get('name')}' missing 'type'")
        if "size" not in room:
            # Beri default size
            room_type = room.get("type", "bedroom")
            defaults = {
                "bedroom": {"width": 12, "height": 15},
                "living_room": {"width": 20, "height": 15},
                "kitchen": {"width": 10, "height": 10},
                "bathroom": {"width": 7, "height": 5},
                "balcony": {"width": 10, "height": 5},
                "common_room": {"width": 15, "height": 12},
                "storage": {"width": 8, "height": 8},
                "dining_room": {"width": 12, "height": 10}
            }
            room["size"] = defaults.get(room_type, {"width": 10, "height": 10})
        
        # Pastikan size punya width dan height
        if "width" not in room["size"] or "height" not in room["size"]:
            room["size"] = {"width": 10, "height": 10}
    
    # Generate summary jika tidak ada
    if "summary" not in data:
        summary = {}
        for room in data["rooms"]:
            room_type = room.get("type", "common_room")
            summary[room_type] = summary.get(room_type, 0) + 1
        data["summary"] = summary
    
    return data, input_tokens, output_tokens

# ============================================================
# FastAPI App
# ============================================================
app = FastAPI(title="Encoder Server", description="Qwen2.5-14B 4-bit - Multilingual Room Extraction")

class EncodeRequest(BaseModel):
    user_text: str

class EncodeResponse(BaseModel):
    status: str
    json_output: str
    token_count_input: int
    token_count_output: int
    token_reduction_percent: float

@app.post("/encode_detailed")
async def encode_detailed(request: EncodeRequest):
    """
    Endpoint utama: menghasilkan JSON lengkap dengan detail ruangan
    (nama, ukuran, lokasi, links) untuk ChatHouseDiffusion.
    Multilingual: mendukung input bahasa Inggris, Indonesia, atau campuran.
    """
    try:
        detailed_json, input_tokens, output_tokens = try_llm_extract_detailed(request.user_text)
        
        final_json_str = json.dumps(detailed_json, ensure_ascii=False, indent=2)
        
        input_chars = len(request.user_text)
        output_chars = len(final_json_str)
        reduction = (1 - output_chars / input_chars) * 100 if input_chars > 0 else 0
        
        return EncodeResponse(
            status="success",
            json_output=final_json_str,
            token_count_input=input_tokens,
            token_count_output=output_tokens,
            token_reduction_percent=round(reduction, 2)
        )
    except Exception as e:
        print(f"[FAILED] Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_ID, "device": str(model.device)}

@app.get("/")
async def root():
    return {"message": "Encoder Server (Qwen2.5-14B-4bit) - Multilingual Room Extraction"}

# --- Start with ngrok ---
print("="*60)
print("[CONNECTING] Connecting to ngrok...")
try:
    ngrok.kill()
    time.sleep(1)
except:
    pass

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
port = 8000
public_url = ngrok.connect(port)
print(f"[SUCCESS] Public URL: {public_url}")
print("="*60)
print(f"[SUCCESS] Main endpoint: {public_url}/encode_detailed")
print("="*60)

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("[SUCCESS] FastAPI server running in background!")
print(f"[SUCCESS] Docs: {public_url}/docs")
print(f"[SUCCESS] Health: {public_url}/health")
print("="*60)
print("[SUCCESS] To stop server, run: ngrok.kill()")

[STARTING] Encoder Server (Qwen2.5-14B-Instruct 4-bit) - MULTILINGUAL ROOM EXTRACTION
[SERVER] Port 8000 cleared.
[SERVER] ngrok token loaded
[LOAD] Loading model Qwen/Qwen2.5-14B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[SUCCESS] Model loaded successfully!
   - Device: cuda:0
   - Parameters: 8.16B
[CONNECTING] Connecting to ngrok...
[SUCCESS] Public URL: NgrokTunnel: "https://smartly-conch-compost.ngrok-free.dev" -> "http://localhost:8000"
[SUCCESS] Main endpoint: NgrokTunnel: "https://smartly-conch-compost.ngrok-free.dev" -> "http://localhost:8000"/encode_detailed
[SUCCESS] FastAPI server running in background!
[SUCCESS] Docs: NgrokTunnel: "https://smartly-conch-compost.ngrok-free.dev" -> "http://localhost:8000"/docs
[SUCCESS] Health: NgrokTunnel: "https://smartly-conch-compost.ngrok-free.dev" -> "http://localhost:8000"/health
[SUCCESS] To stop server, run: ngrok.kill()


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔍 [DEBUG] Raw LLM output:
{
  "rooms": [
    {"name": "balcony", "type": "balcony", "size": {"width": 12, "height": 5}, "location": "south", "links": ["master room"]},
    {"name": "master room", "type": "bedroom", "size": {"width": 12, "height": 20}, "location": "east", "links": ["balcony", "common room", "living room", "kitchen"]},
    {"name": "bathroom", "type": "bathroom", "size": {"width": 12, "height": 5}, "location": "northeast corner", "links": ["common room", "living room"]},
    {"name": "common room", "type": "common_room", "size": {"width": 12, "height": 10}, "location": "east side", "links": ["bathroom", "master room", "living room"]},
    {"name": "kitchen", "type": "kitchen", "size": {"width": 10, "height": 10}, "location": "southwest corner", "links": ["living room", "master room"]},
    {"name": "living room", "type": "living_room", "size": {"width": 15, "height": 30}, "location": "northwest corner", "links": ["bathroom", "common room", "master room", "kitchen"]}
  ],